# Step A: retrain the clean baseline M0, properly

The pilot diagnostic showed the v1 models draw tumor on **every** tumor-free slice (they were trained on the 10 largest-tumor slices only), so their true full-volume 3D Dice is ~0.2, not the ~0.75 the training logs reported. This notebook retrains **M0** with the fixes:

- all brain slices, tumor-free ones included
- validation = full-volume 3D Dice on all 37 validation patients, plus the false-tumor volume on tumor-free slices
- batch-level soft Dice + BCE, 30 epochs, mixed precision

**Success looks like:** validation 3D Dice well above 0.2 (a well-trained 2D U-Net on BraTS typically reaches roughly 0.7-0.8 mean over WT/TC/ET), and false tumor on tumor-free slices close to 0 cm³ (v1: ~450 cm³).

**Before Run all:** `archive.zip` at `MyDrive/Argus/archive.zip` (already there from the pilot); *Runtime → Change runtime type → GPU* (an A100 or L4 is much faster than a T4 for training). Checkpoints are saved to Drive after every epoch: if Colab disconnects, run all again and training resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- Configuration ----
ZIP   = '/content/drive/MyDrive/Argus/archive.zip'
OUT   = '/content/drive/MyDrive/Argus/train_v2'     # checkpoints + logs (resumable)
WORK  = '/content/cache'                            # prepared arrays (fast local disk)
MODEL, SEED, EPOCHS = 'M0', 0, 30
REPO, BRANCH = 'https://github.com/joshykoshy/Argus.git', 'matlab-analysis'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
import os
if os.path.isdir('/content/Argus'):
    !git -C /content/Argus pull -q
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Argus
%cd /content/Argus
!pip -q install nibabel

In [ ]:
# Extract the 258 training + 37 validation patients (~25 GB, 10-20 min). Skipped if already done.
import json, zipfile, glob
split = json.load(open('data/splits/split_v1.json'))
need = split['train'] + split['val']
if not glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True):
    with zipfile.ZipFile(ZIP) as z:
        names = [n for n in z.namelist() if any(f'/{p}/' in n for p in need) or n.endswith('/name_mapping.csv')]
        z.extractall('/content/brats', members=names)
RAW = glob.glob('/content/brats/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
print(RAW, len([d for d in os.listdir(RAW) if d.startswith('BraTS20')]), 'patients')

In [ ]:
# Preprocess into memory-mappable arrays (~15-25 min, resumable), then free the raw files
!python -m training.prepare --raw_dir {RAW} --work {WORK}
!rm -rf /content/brats
!du -sh {WORK}

In [ ]:
# Smoke test: 1 tiny epoch + validation on 2 patients (checks everything runs, shows speed)
!rm -rf /content/smoke
!python -m training.train_v2 --model {MODEL} --work {WORK} --out /content/smoke --epochs 1 --slices_per_epoch 640 --val_patients 2

In [ ]:
# Full training (resumes from Drive if interrupted)
!python -m training.train_v2 --model {MODEL} --seed {SEED} --epochs {EPOCHS} --work {WORK} --out {OUT}

In [ ]:
# Learning curve
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/{MODEL}/seed_{SEED}/log.csv')
v = log.dropna(subset=['val_dice_3d'])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(log.epoch, log.train_loss, 'o-'); ax[0].set_title('training loss'); ax[0].set_xlabel('epoch')
ax[1].plot(v.epoch, v.val_dice_3d, 'o-'); ax[1].axhline(0.21, ls=':', c='gray'); ax[1].set_title('validation full-volume 3D Dice (v1 M0: 0.21)'); ax[1].set_xlabel('epoch')
ax[2].plot(v.epoch, v.val_false_tumor_cm3, 'o-'); ax[2].set_title('false tumor on tumor-free slices (cm³, v1: ~450)'); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{OUT}/{MODEL}/seed_{SEED}/learning_curve.png', dpi=130); plt.show()
print(log.tail(5).to_string(index=False))

**What to bring back:** the last cell's table, or just say "done": `log.csv` and `learning_curve.png` are in `MyDrive/Argus/train_v2/M0/seed_0/`, and the trained model is `best_model.pt` in the same folder. Next is Step B: rerunning the pilot with this M0 to choose the degradation range.